# GTEx Pathology Annotations for Tissue Slides in IDC

The [GTEx](https://gtexportal.org) collection in IDC (`gtex`) contains 25,503 H&E whole-slide images from 971 healthy postmortem donors across 50+ tissue types.
Each slide was reviewed and annotated by a board-certified pathologist.

**Source:** GTEx portal API (`https://gtexportal.org/api/v2/biobank/sample`) — not GDC.
No PDF reports exist; annotations are returned as structured JSON.

**Coverage:** 22,909 of 25,503 IDC tissue samples (89.8%) matched; 100% of those have pathology notes.

## Annotation types

| Field | Type | Content |
|-------|------|---------|
| `pathologyNotes` | Free text | Specimen quality notes — piece count, autolysis, congestion, special findings |
| `pathologyNotesCategories` | Dict of 57 booleans | Structured pathological findings (atherosclerosis, fibrosis, congestion, …) |

## Join key

| IDC field | GTEx API field | How to map |
|-----------|----------------|-----------|
| `sm_index.ContainerIdentifier` | `biobank/sample.tissueSampleId` | See note below |

**ContainerIdentifier format note:**

99% of IDC ContainerIdentifiers use the format `GTEX-DONOR-XYZS5` (4-digit suffix ending in `5`).
These map to the GTEx API `tissueSampleId` `GTEX-DONOR-XYZS6` — same prefix, last digit `5→6`.

Example: `GTEX-1117F-1025` (IDC) ↔ `GTEX-1117F-1026` (API)

The remaining ~1% (263 slides) use a different schema (`GTEX-DONOR-5XXX`) that does not appear
in the current GTEx biobank API and are therefore unmatched.

## Requirements

```
pip install idc-index requests pandas
```

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from idc_index import IDCClient

GTEX_SAMPLE_API = "https://gtexportal.org/api/v2/biobank/sample"

client = IDCClient()
client.fetch_index("sm_index")
print("IDC version:", client.get_idc_version())

## 1. Get Tissue Sample IDs from IDC

`sm_index.ContainerIdentifier` holds the GTEx tissue sample ID as recorded on the slide (e.g., `GTEX-1117F-1025`).
`PatientID` (e.g., `GTEX-1117F`) is the donor/subject ID.
Both come from the sm_index joined to the primary index.

In [ ]:
idc_samples = client.sql_query("""
    SELECT DISTINCT
        i.PatientID                                    AS donor_id,
        s.ContainerIdentifier                          AS container_id,
        s.primaryAnatomicStructure_CodeMeaning         AS tissue,
        s.SeriesInstanceUID
    FROM sm_index s
    JOIN index i USING (SeriesInstanceUID)
    WHERE i.collection_id = 'gtex'
    ORDER BY i.PatientID, s.ContainerIdentifier
""")

print(f"IDC GTEx tissue samples : {len(idc_samples):,}")
print(f"Unique donors           : {idc_samples['donor_id'].nunique()}")
print(f"Unique tissues          : {idc_samples['tissue'].nunique()}")
print()
print(idc_samples.head(6).to_string(index=False))

## 2. Fetch Pathology Annotations from GTEx API

The GTEx portal API returns all biobank sample records including pathology notes.
Pagination is required (250 records/page, ~303 pages for the full dataset).

**Runtime:** ~2 minutes to download all 75,568 sample records.

In [ ]:
def fetch_all_gtex_samples(page_size: int = 250, cache_path: Path | None = None) -> dict[str, dict]:
    """
    Download all GTEx biobank sample records and return a dict keyed by tissueSampleId.

    If cache_path is given, saves/loads a JSON cache to avoid re-downloading.
    """
    if cache_path and Path(cache_path).exists():
        print(f"Loading cache: {cache_path}")
        return json.loads(Path(cache_path).read_text())

    samples_by_id = {}
    page = 0
    while True:
        r = requests.get(GTEX_SAMPLE_API,
            params={"pageSize": page_size, "page": page}, timeout=30)
        r.raise_for_status()
        data = r.json()
        for s in data["sample"]:
            samples_by_id[s["tissueSampleId"]] = s
        if page >= data["numPages"] - 1:
            break
        page += 1
        if page % 50 == 0:
            print(f"  page {page}/{data['numPages']}  ({len(samples_by_id):,} records)")

    print(f"Total records fetched: {len(samples_by_id):,}")
    if cache_path:
        Path(cache_path).write_text(json.dumps(samples_by_id))
        print(f"Cached to: {cache_path}")
    return samples_by_id


# ~2 min first run; instant on subsequent runs with cache
api_samples = fetch_all_gtex_samples(
    cache_path="/tmp/gtex_api_samples.json"
)

## 3. Join IDC Slides to Pathology Annotations

**99% of IDC ContainerIdentifiers** use the format `GTEX-DONOR-XYZS5`. These map to the GTEx API
`tissueSampleId` `GTEX-DONOR-XYZS6` (same prefix, last digit `5→6`).

The remaining ~1% use a different schema (`GTEX-DONOR-5XXX`) not present in the current API.
They are included in the output with null annotation fields.

In [ ]:
def gtex_api_id(container_id: str) -> str:
    """Convert IDC ContainerIdentifier to GTEx API tissueSampleId."""
    return container_id[:-1] + "6"


rows = []
for _, row in idc_samples.iterrows():
    api_id = gtex_api_id(row["container_id"])
    api = api_samples.get(api_id)
    rows.append({
        "donor_id"          : row["donor_id"],
        "container_id"      : row["container_id"],
        "api_sample_id"     : api_id,
        "tissue_idc"        : row["tissue"],
        "tissue_api"        : api["tissueSiteDetail"] if api else None,
        "sex"               : api["sex"]              if api else None,
        "age_bracket"       : api["ageBracket"]       if api else None,
        "hardy_scale"       : api["hardyScale"]       if api else None,
        "autolysis_score"   : api["autolysisScore"]   if api else None,
        "pathology_notes"   : api["pathologyNotes"]   if api else None,
        "pathology_cats"    : api["pathologyNotesCategories"] if api else None,
        "SeriesInstanceUID" : row["SeriesInstanceUID"],
    })

df = pd.DataFrame(rows)
matched   = df["tissue_api"].notna()
has_notes = df["pathology_notes"].fillna('').str.len() > 0

print(f"Joined records        : {len(df):,}")
print(f"Matched in API        : {matched.sum():,} ({matched.mean()*100:.1f}%)")
print(f"Has pathology notes   : {has_notes.sum():,} ({has_notes.mean()*100:.1f}%)")
print()
print(df[matched].head(4)[["donor_id","tissue_api","age_bracket","hardy_scale","pathology_notes"]].to_string(index=False))

## 4. Structured Pathology Categories

The `pathologyNotesCategories` dict contains up to 56 boolean flags for specific pathological findings.
Categories are tissue-type specific — e.g., `spermatogenesis` only applies to testis,
`glomerulosclerosis` to kidney, `emphysema` to lung.

~22% of matched samples have at least one True category; the rest have all-False (typical for clean specimens).

In [ ]:
from collections import Counter

# Expand pathology_cats column into boolean columns
cat_dicts = df.loc[matched & df["pathology_cats"].notna(), "pathology_cats"]

# Get full list of category keys
all_cats = sorted({k for d in cat_dicts for k in d.keys()})
print(f"Pathology category keys ({len(all_cats)}):")
print(all_cats)

# Expand to boolean DataFrame
cats_df = pd.DataFrame(list(cat_dicts), index=cat_dicts.index).fillna(False).astype(bool)

# Prevalence across all matched samples with categories
prevalence = cats_df.sum().sort_values(ascending=False)
print(f"\nTop 20 most common True categories (out of {len(cats_df):,} samples with categories):")
print(prevalence.head(20).to_string())

In [ ]:
from collections import Counter

# Expand pathology_cats column into boolean columns
cat_dicts = df.loc[matched & df["pathology_cats"].notna(), "pathology_cats"]

# Get full list of category keys
all_cats = sorted({k for d in cat_dicts for k in d.keys()})
print(f"Pathology category keys ({len(all_cats)}):")
print(all_cats)

# Expand to boolean DataFrame
cats_df = (
    pd.DataFrame(list(cat_dicts), index=cat_dicts.index)
    .infer_objects(copy=False)
    .fillna(False)
    .astype(bool)
)

# Prevalence across all matched samples with categories
prevalence = cats_df.sum().sort_values(ascending=False)
print(f"\nTop 20 most common True categories (out of {len(cats_df):,} samples with categories):")
print(prevalence.head(20).to_string())

## 5. Donor Metadata

The API also provides basic donor metadata per tissue sample:
- `sex`: male / female
- `ageBracket`: age range (20-29, 30-39, …, 70-79)
- `hardyScale`: death circumstance — Ventilator case / Fast death – violent / Fast death – natural causes / Intermediate / Slow death
- `autolysisScore`: tissue quality — None / Mild / Moderate / Severe

In [ ]:
matched_df = df[matched].copy()

print("Sex distribution:")
print(matched_df.drop_duplicates("donor_id")["sex"].value_counts().to_string())

print("\nAge bracket distribution:")
print(matched_df.drop_duplicates("donor_id")["age_bracket"].value_counts().sort_index().to_string())

print("\nHardy scale (death circumstance):")
print(matched_df.drop_duplicates("donor_id")["hardy_scale"].value_counts().to_string())

print("\nAutolysis score (tissue quality):")
print(matched_df["autolysis_score"].value_counts(dropna=False).to_string())

## 6. Per-Donor Lookup

Fetch annotations for a single donor, merge with IDC viewer URLs.

In [ ]:
DONOR = "GTEX-1117F"

donor_df = df[df["donor_id"] == DONOR].copy()
print(f"{DONOR} — {len(donor_df)} tissue slides")
print()

for _, row in donor_df.iterrows():
    url = client.get_viewer_URL(seriesInstanceUID=row["SeriesInstanceUID"])
    notes = row.get("pathology_notes") or "—"
    # Flag any non-clean categories
    cats = row.get("pathology_cats") or {}
    findings = [k for k, v in cats.items() if v and k != "clean_specimens"]
    print(f"  {row['tissue_api'] or row['tissue_idc']:40s}  {notes[:60]}")
    if findings:
        print(f"    findings: {findings}")
    # Uncomment to open in browser:
    # import webbrowser; webbrowser.open(url)

## 7. Save Joined Annotations as CSV

Export the full join as a flat CSV for downstream analysis.

In [ ]:
out_path = Path("/tmp/gtex_pathology_annotations.csv")

# Expand pathology_cats dict into flat boolean columns
cats_expanded = pd.DataFrame(
    df["pathology_cats"].apply(lambda x: x if isinstance(x, dict) else {}).tolist(),
    index=df.index
).fillna(False).astype(bool)

out_df = pd.concat([
    df[["donor_id", "container_id", "api_sample_id", "tissue_api",
        "sex", "age_bracket", "hardy_scale", "autolysis_score",
        "pathology_notes", "SeriesInstanceUID"]],
    cats_expanded
], axis=1)

out_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {out_df.shape}  ({out_df.shape[1]} columns = 10 metadata + {cats_expanded.shape[1]} category flags)")
print(out_df.head(3).to_string(index=False))